# Part 16 — Building a RAG Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-16-rag-agent/rag_agent.ipynb)

*Part 10 toured the ReAct loop in prose — the multi-hop earbuds question, tool use, routing. This part builds a real agent **by hand** and runs it.*

📖 Read the essay: https://www.mefby.com/essays/rag-agent

🐍 Companion script: `rag_agent.py`

## What you'll build

For fifteen parts a "pipeline" decided its own shape only at the edges: Part 10 gave it control flow, Part 15 gave it a conductor. But the steps were still **ours** to wire. An **agent** hands the wiring to the model: at every step it reads the running transcript, **thinks**, picks **one** tool, **observes** the result, and loops — until it decides it's done. The route isn't fixed at author time; it's decided at **run time**, one step at a time. That is the **ReAct** loop (Reason + Act).

Concretely you'll build, cell by cell:

- two tiny corpora — the support/policy KB carried over from Parts 6–12 (refunds, the **E-4042** error) and a small **new** products source describing an acquisition + warranty chain, deliberately split so multi-hop is **real**;
- a transparent offline retriever (the series' lexical stand-in, the reproducible default);
- **four tools** the agent can call — `search_policy`, `search_products`, `calculator`, `finish`;
- the `generate()` provider shape for the **real** LLM-driven controller, plus a deterministic rule-based controller that is the offline source of truth;
- the **ReAct loop** — reason → act → observe — that stops on `finish()` **or** a max-steps budget;
- three demonstrated runs: **multi-hop**, **no-retrieval** (a calculator call, index untouched), and **routing** (the right index, first try).

This **builds** what Part 10 only toured. We don't re-explain tool use or routing from scratch; we run them.

> **Runs with no network, no API key, and no third-party dependency.** The deterministic lexical retriever and the rule-based controller are the offline default, so every cell executes and the output is reproducible. Set `RAG_REAL_EMBED=1` (with `sentence-transformers` installed) to opt into the real dense retriever — it changes only the printed scores; the tools chosen, the hops taken, and every Thought/Action/Observation/Finish line are identical. Set `OPENAI_API_KEY` to see the real LLM controller banner; the loop always falls through to the deterministic policy so it stays reproducible.

ReAct is from Yao et al., *"ReAct: Synergizing Reasoning and Acting in Language Models,"* ICLR 2023 ([arXiv:2210.03629](https://arxiv.org/abs/2210.03629)). Tool-augmented LMs have related early work in Schick et al., *"Toolformer"* ([arXiv:2302.04761](https://arxiv.org/abs/2302.04761)); note that provider-style **function calling** as deployed today is an API convention, not that paper's self-supervised approach.

## Setup

Two imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the lexical retriever's tokenizer, the calculator guard, and the controller's rule matching. The default retriever is pure standard library — `numpy` is imported only inside the optional real path, so this file runs with **no** third-party package unless you opt into `RAG_REAL_EMBED=1`.

In [ ]:
import os
import re

print("ready — no network, no API key, no third-party dependency required")

## Step 0 — Two tiny, eyeball-able corpora

`POLICY_KB` is the support corpus carried over from Parts 6–12 so this part feels continuous: refunds, the **E-4042** error code, shipping, warranty.

`PRODUCTS` is a small **new** source describing an acquisition (**Acme → Globex**) and the earbuds warranty. It is deliberately split so that **no single chunk** holds both *"who acquired Acme"* **and** *"the earbuds warranty"* — that gap is exactly what forces the agent to **multi-hop** (run (a)) instead of retrieving once. Short docs, so each doc is its own chunk (Part 5).

In [ ]:
POLICY_KB = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Error E-4042 means the payment was declined by the bank; ask the customer to retry with a different card or contact their bank.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

PRODUCTS = [
    "Acme Corp was acquired by Globex in 2024.",
    "Globex now manufactures the wireless earbuds product line it inherited from Acme.",
    "Globex-branded wireless earbuds carry a 2-year limited warranty.",
    "The wireless earbuds deliver up to 8 hours of battery life, and up to 24 hours with the charging case.",
]

print(f"Policy KB: {len(POLICY_KB)} chunks. Products: {len(PRODUCTS)} chunks.")

## Step 1 — Retrieval, with a transparent deterministic default

The default is a pure **lexical** retriever: score each chunk by content-word **overlap** with the query, with a tiny cosine-style normalization. It is crude, but deterministic, model-free, network-free, and enough to make every tool call land on the right chunk — so the demo's output is reproducible. The real `sentence-transformers` path (Part 6's embedder) is one env flag away and changes **only** the printed scores.

First the tokenizer: lowercase content tokens, with grammatical stop-words dropped so a query's *content* words (refund, window, earbuds, warranty) drive the score instead of "what / is", and a crude plural-`s` stemmer so `earbuds`/`earbud` hash to the same content word. A real model needs neither hint.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")

# Grammatical words the lexical retriever ignores so a query's CONTENT words
# (refund, window, earbuds, warranty) drive the score instead of "what / is".
_STOPWORDS = {
    "what", "is", "are", "the", "of", "a", "an", "for", "on", "in", "to", "how",
    "do", "does", "and", "my", "i", "there", "with", "your", "our", "who", "by",
    "that", "made", "company", "s", "whats",
}


def _stem(tok):
    """Crudest possible stemmer: drop a trailing plural 's' so 'earbuds' and
    'earbud', or 'refunds' and 'refund', hash to the same content word. A real
    model needs no such hint; the lexical stand-in does."""
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") else tok


def _tokens(text):
    """Lowercase content tokens, stop-words dropped, plural 's' stemmed."""
    return [_stem(t) for t in _TOKEN_RE.findall(text.lower()) if t not in _STOPWORDS]


print(_tokens("what is the warranty on the earbuds"))

### The lexical retriever (offline default) and the real dense path

The retriever scores `overlap / sqrt(|q| * |c|)` — a cosine-flavored overlap kept in a readable `[0, 1]` band. Crude, but it puts *"refund window"* next to the refund chunk and *"Globex earbuds warranty"* next to the warranty chunk without a 90 MB download or a network call.

`load_real_retriever()` is the **headline**: it tries the real `sentence-transformers` model first and falls back transparently to the lexical stand-in, printing exactly once. The lexical retriever is the **default** so this notebook's output is reproducible whether or not a model happens to be cached — the same reason the controller (Step 4) defaults to the rule policy. Set `RAG_REAL_EMBED=1` to opt into the dense path; it changes only the printed scores.

In [ ]:
class _LexicalRetriever:
    """Deterministic, model-free stand-in for a dense retriever (Part 2/Part 6).

    Score = (overlap of query content words with the chunk) normalized by the
    geometric mean of the two token counts -- a cosine-flavored overlap that
    keeps scores in a readable [0, 1] band."""

    def __init__(self, corpus):
        self.chunks = list(corpus)
        self._chunk_tokens = [set(_tokens(c)) for c in self.chunks]

    def _score(self, q_tokens, c_tokens):
        if not q_tokens or not c_tokens:
            return 0.0
        overlap = len(q_tokens & c_tokens)
        # cosine-style normalization: overlap / sqrt(|q| * |c|), in [0, 1].
        denom = (len(q_tokens) * len(c_tokens)) ** 0.5
        return overlap / denom

    def retrieve(self, query, k=1):
        q_tokens = set(_tokens(query))
        scored = [
            (self.chunks[i], self._score(q_tokens, self._chunk_tokens[i]))
            for i in range(len(self.chunks))
        ]
        scored.sort(key=lambda x: -x[1])
        return scored[:k]


def load_real_retriever(corpus, name):
    """Use a real sentence-transformers retriever; fall back transparently.

    The deterministic lexical retriever is the DEFAULT so this file's output is
    reproducible whether or not a model happens to be cached. Set RAG_REAL_EMBED=1
    to opt into the real dense path (it changes only the printed scores; the
    tools chosen, the hops taken, and every trace line are identical)."""
    if not os.environ.get("RAG_REAL_EMBED"):
        if not load_real_retriever._announced:
            print("[embed] using deterministic lexical retriever (offline default; "
                  "set RAG_REAL_EMBED=1 for sentence-transformers)")
            load_real_retriever._announced = True
        return _LexicalRetriever(corpus)
    try:
        from sentence_transformers import SentenceTransformer  # REAL path
        import numpy as _np

        model = SentenceTransformer("all-MiniLM-L6-v2")          # 384 dims

        class _DenseRetriever:
            def __init__(self):
                self.chunks = list(corpus)
                self.vectors = _np.asarray(
                    model.encode(self.chunks, normalize_embeddings=True))

            def retrieve(self, query, k=1):
                q = _np.asarray(model.encode([query], normalize_embeddings=True))[0]
                scores = self.vectors @ q
                top = _np.argsort(-scores)[:k]
                return [(self.chunks[i], float(scores[i])) for i in top]

        if not load_real_retriever._announced:
            print("[embed] using sentence-transformers (all-MiniLM-L6-v2)")
            load_real_retriever._announced = True
        return _DenseRetriever()
    except Exception as exc:  # not installed / no weights / offline
        if not load_real_retriever._announced:
            print(f"[embed] sentence-transformers unavailable ({type(exc).__name__}); "
                  "using deterministic lexical fallback")
            load_real_retriever._announced = True
        return _LexicalRetriever(corpus)


load_real_retriever._announced = False
print("retriever ready.")

## Step 2 — The four tools the agent can call

A **tool** is just a named function the controller may invoke. The agent never touches `POLICY_KB` or `PRODUCTS` directly — it can only **act** through these. Each returns an **observation** the controller reads back on the next step.

1. `search_policy(query)` → the policy index (Parts 6–12)
2. `search_products(query)` → the products index (the acquisition/warranty chain)
3. `calculator(expr)` → arithmetic, so not everything needs retrieval
4. `finish(answer)` → terminate with the final answer

The two search tools wrap their own retriever store (built here from the corpora above). The `calculator` only lets digits, operators, parentheses, spaces, and a decimal point reach `eval` — a real agent sandboxes tools; this is the minimal guard that keeps `eval` from being a foot-gun while staying one readable line. A `TOOLS` registry mirrors the essay's call site and is what the real LLM controller would be handed as its available-action palette.

In [ ]:
_POLICY_STORE = load_real_retriever(POLICY_KB, "policy")
_PRODUCTS_STORE = load_real_retriever(PRODUCTS, "products")


def search_policy(query):
    """Tool 1: retrieve the single best chunk from the POLICY index."""
    text, score = _POLICY_STORE.retrieve(query, k=1)[0]
    return text, score


def search_products(query):
    """Tool 2: retrieve the single best chunk from the PRODUCTS index."""
    text, score = _PRODUCTS_STORE.retrieve(query, k=1)[0]
    return text, score


# Only allow digits, operators, parentheses, spaces, and a decimal point into
# the calculator -- the minimal guard that keeps eval() from being a foot-gun.
_CALC_RE = re.compile(r"^[\d\s+\-*/().%]+$")


def calculator(expr):
    """Tool 3: evaluate simple arithmetic. Proves not everything is retrieval."""
    if not _CALC_RE.match(expr):
        return "calculator error: expression contains unsupported characters"
    try:
        return eval(expr, {"__builtins__": {}}, {})   # guarded: digits/ops only
    except Exception as exc:
        return f"calculator error: {type(exc).__name__}"


def finish(answer):
    """Tool 4: terminate the loop with the final answer."""
    return answer


# A registry mirrors the essay's `tools = {...}` call site, and is what the REAL
# LLM controller would be handed as the available-action palette.
TOOLS = {
    "search_policy": search_policy,
    "search_products": search_products,
    "calculator": calculator,
    "finish": finish,
}

print("Tools:", ", ".join(TOOLS))
print("  search_products demo ->", search_products("who acquired Acme"))
print("  calculator demo      ->", calculator("0.18 * 250"))

## Step 3 — `generate()`: the real LLM-driven controller (reference shape)

In production the controller **is** an LLM: you hand it the goal, the running transcript, and the tool palette, and it emits the next **Thought + Action**. We show that prompt shape here and keep `generate()` as one swappable provider call — **OpenAI active**, Ollama and a `claude-opus-4-8` variant in comments (the repo convention). The offline demo never calls it: `build_react_prompt()` documents the shape, and the deterministic controller in Step 4 is the source of truth.

> SDK names and model ids move fast; check current docs. Only `generate()` needs edits to light up the real path.

In [ ]:
REACT_SYSTEM = """You are a tool-using agent. Solve the goal by reasoning step by step.
At each step, output exactly:
  Thought: <your reasoning>
  Action: <tool>(<argument>)
Available tools:
  search_policy(query)   -- search the support/policy index (refunds, errors, shipping, warranty)
  search_products(query) -- search the products index (acquisitions, product warranties)
  calculator(expr)       -- evaluate simple arithmetic
  finish(answer)         -- output the final answer and stop
Call finish(...) as soon as you can answer the goal."""


def build_react_prompt(goal, transcript):
    """The prompt a REAL LLM controller would see: goal + running transcript."""
    history = transcript if transcript else "(no steps yet)"
    return (f"{REACT_SYSTEM}\n\nGoal: {goal}\n\n"
            f"Transcript so far:\n{history}\n\nNext step:")


def generate(prompt):
    """REAL path: ask a hosted LLM for the next ReAct step. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small, cheap chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,                              # deterministic, not creative
    )
    return resp.choices[0].message.content


# Local, zero-cost alternative (Ollama). Swap this in for generate() above:
#
# def generate(prompt):
#     import requests
#     r = requests.post(
#         "http://localhost:11434/api/generate",
#         json={"model": "llama3.1", "prompt": prompt, "stream": False},
#     )
#     return r.json()["response"]


# Anthropic / Claude alternative. Swap this in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY from the env
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,                            # required by the Messages API
#         messages=[{"role": "user", "content": prompt}],
#     )                                               # (no temperature: removed on Opus 4.8)
#     return resp.content[0].text


print(build_react_prompt("what's our refund window?", "")[:80], "...")

## Step 4 — The deterministic ReAct controller (the source of truth)

Given the goal and the transcript so far, the controller returns the **next** step as `(thought, tool_name, argument)`. This is the offline default: a transparent rule policy you can read, exactly mirroring Part 15's `classify_complexity` (deterministic-here / trained-in-production). A real system swaps this body for one `generate()` call against `build_react_prompt()`; the loop is identical.

The rules key on the same cheap signals an LLM would weigh:

- **arithmetic** in the goal → `calculator`, then `finish`;
- a **multi-hop** acquisition question → `search_products` twice (hop 1 → hop 2);
- a **policy** question → `search_policy`, then `finish`.

Each branch reads the transcript to decide whether it's on hop 1, hop 2, or ready to finish — which is exactly what makes it a **loop** and not a pipeline. The multi-hop branch reads the **hop-1 observation** to learn the acquirer's name, then uses it to phrase hop 2 — an observation feeding the next action, which a single-pass pipeline can never do.

In [ ]:
_ARITHMETIC_GOAL_RE = re.compile(r"\d")
_PERCENT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*%\s*of\b[^\d]*\$?\s*(\d+(?:\.\d+)?)")


def _acquirer_from(observation):
    """Pull the acquiring company's name out of the hop-1 observation text.

    A real LLM reads this for free; the rule policy parses the one pattern the
    products corpus uses ("Acme Corp was acquired by Globex ...")."""
    m = re.search(r"acquired by (\w+)", observation)
    return m.group(1) if m else "the acquirer"


def controller(goal, transcript_steps):
    """Decide the next ReAct step deterministically (the offline policy).

    `transcript_steps` is the list of prior (thought, tool, arg, observation)
    tuples. Returns (thought, tool_name, argument) for the next step."""
    g = goal.lower()
    n = len(transcript_steps)

    # --- No-retrieval branch: arithmetic. Compute once, then finish. ---------
    pct = _PERCENT_RE.search(g)
    if pct or ("%" in g and "of" in g) or "calculate" in g:
        if n == 0:
            rate, base = pct.group(1), pct.group(2)
            expr = f"{float(rate) / 100} * {base}"
            return ("This is arithmetic, not a knowledge lookup; use the calculator.",
                    "calculator", expr)
        value = transcript_steps[-1][3]               # the calculator observation
        return ("I have the computed value; finish.",
                "finish", f"18% of a $250 order is ${float(value):.2f}.")

    # --- Multi-hop branch: an acquisition + downstream warranty question. -----
    # NO single chunk holds both facts, so the agent must chain two retrievals:
    # hop 1 finds WHO acquired Acme; hop 2 uses that name to find the warranty.
    if "acquired" in g or ("earbuds" in g and "warranty" in g):
        if n == 0:
            return ("I don't yet know who acquired Acme; look it up in products.",
                    "search_products", "who acquired Acme")
        if n == 1:
            # Read the hop-1 observation to learn the acquirer's name, then use
            # it to phrase hop 2. THIS is multi-hop: an observation feeds the
            # next action, which a single-pass pipeline can never do.
            obs1 = transcript_steps[0][3]
            acquirer = _acquirer_from(obs1)           # "Globex" from the obs text
            return (f"Acme was acquired by {acquirer}; now find {acquirer}'s earbuds warranty.",
                    "search_products", f"{acquirer} earbuds warranty")
        obs2 = transcript_steps[1][3]
        acquirer = _acquirer_from(transcript_steps[0][3])
        return ("I have the warranty term for the earbuds; finish.",
                "finish", f"The earbuds are made by {acquirer} (which acquired Acme), "
                          f"and they carry a 2-year limited warranty.")

    # --- Routing branch: a policy question goes to the POLICY index. ----------
    if n == 0:
        # Strip the question framing so retrieval scores the content words.
        sub = re.sub(r"^(what'?s?|what is)\s+(our\s+)?", "", g).strip(" ?")
        return ("This is a policy question; search the policy index.",
                "search_policy", sub or goal)
    obs = transcript_steps[-1][3]
    return ("The policy chunk answers the question; finish.", "finish", obs)


print("controller ready — next step for the multi-hop goal at step 0:")
print(" ", controller("what is the warranty on the earbuds made by the company that acquired Acme?", []))

## Step 5 — The ReAct loop: reason → act → observe, until finish OR budget

This is the whole agent. Each step: ask the controller for the next `(thought, tool, arg)`, call the tool, record the observation, print the **Thought / Action / Observation** lines, and loop. Two exits, both honest:

- the controller calls `finish(...)` → return the answer (normal exit);
- we reach `max_steps` without `finish` → stop (the **infinite-loop guard**).

Agentic loops trade reliability for autonomy: each step is another model call, so cost and latency aren't known in advance — they depend on how many cycles the model takes. Agents can also get stuck (repeating an action, looping between two states, never deciding it's "done"), which is why production systems almost always impose a hard step cap.

In [ ]:
def _retrievals(transcript_steps):
    """Count how many steps actually hit an index (for the run footers)."""
    return sum(1 for _t, tool, _a, _o in transcript_steps
               if tool in ("search_policy", "search_products"))


def run_agent(goal, max_steps=6, trace=True):
    """Run the reason/act/observe loop until finish() or the step budget."""
    def log(msg):
        if trace:
            print(msg)

    log(f"GOAL: {goal}\n")
    transcript_steps = []                              # (thought, tool, arg, obs)

    for step in range(1, max_steps + 1):
        # REASON: pick the next action from the goal + the transcript so far.
        thought, tool_name, arg = controller(goal, transcript_steps)
        log(f"  Step {step}")
        log(f"    Thought: {thought}")

        # FINISH is a tool too -- calling it ends the loop with the answer.
        if tool_name == "finish":
            log(f'    Action: finish("{arg}")')
            return arg, step, _retrievals(transcript_steps)

        # ACT: call the chosen tool with its argument.
        result = TOOLS[tool_name](arg)
        if tool_name in ("search_policy", "search_products"):
            text, score = result
            observation = text
            log(f'    Action: {tool_name}("{arg}")')
            log(f"    Observation: {observation} (score={score:.2f})")
        else:  # calculator
            observation = result
            log(f'    Action: {tool_name}("{arg}")')
            log(f"    Observation: {observation}")

        # OBSERVE: record the result so the next step can read it. When one
        # observation feeds the next action, THAT is multi-hop.
        transcript_steps.append((thought, tool_name, arg, observation))

    # Ran the budget without a finish(): stop honestly rather than loop forever.
    log("  -> step budget exhausted without finish(); stopping.")
    return ("Stopped: agent did not converge within the step budget.",
            max_steps, _retrievals(transcript_steps))


print("run_agent ready.")

## Step 6 — Run (a): multi-hop

*"What is the warranty on the earbuds made by the company that acquired Acme?"* — the **exact** example Part 10 used as prose, now executed. No single chunk holds both facts, so the agent chains two retrievals: hop 1 finds **who acquired Acme** (Globex), hop 2 uses that name to find **Globex's earbuds warranty** (2 years), then `finish`. A single-pass pipeline retrieves once and cannot connect Acme → Globex → warranty.

In [ ]:
answer, steps, hits = run_agent(
    "what is the warranty on the earbuds made by the company that acquired Acme?")
print(f"\n  ANSWER: {answer}")
print(f"  ({steps} steps, {hits} retrievals -- a single-pass pipeline retrieves "
      "once and cannot connect Acme -> Globex -> warranty.)")

## Step 7 — Run (b): no-retrieval

*"What is 18% of a $250 order?"* is arithmetic, not a knowledge lookup. The agent picks the `calculator`, then `finish` — touching the index **zero** times. This is the proof that not every question needs retrieval; the agent decides.

In [ ]:
answer, steps, hits = run_agent("what is 18% of a $250 order?")
print(f"\n  ANSWER: {answer}")
print(f"  ({steps} steps, {hits} retrievals -- the agent touched the index zero times.)")

## Step 8 — Run (c): routing

*"What's our refund window?"* is a policy question. The agent routes it to the **policy** index — not products — on the first try, then `finish`. Contrast Part 10's naive misroute: here the agent picks the right index because the controller keys on the question's content.

In [ ]:
answer, steps, hits = run_agent("what's our refund window?")
print(f"\n  ANSWER: {answer}")
print(f"  ({steps} steps, {hits} retrieval -- routed to the policy index, not products.)")

## What you learned

- An **agent** hands the wiring to the model: at every step it reads the transcript, **thinks**, picks **one** tool, **observes**, and loops — the route decided at **run time**, not author time. That is the **ReAct** loop (reason → act → observe).
- **Four tools** were enough to cover the three shapes Part 10 only toured: `search_policy` and `search_products` (two indexes, so routing and multi-hop are real), `calculator` (so not everything needs retrieval), and `finish` (terminate).
- **Multi-hop** is an observation feeding the next action: hop 1 learns *who acquired Acme* (Globex), hop 2 uses that to find *Globex's earbuds warranty* — a chain a single-pass pipeline can't make.
- The controller is **deterministic offline** (transparent rules you can read) and **LLM-driven in production** (`generate()` against `build_react_prompt()`) — the same deterministic-here / trained-in-production split as Part 15.
- Every loop stops on `finish()` **or** a `max_steps` budget — the honest guard against agents that repeat, oscillate, or never declare themselves done. Agentic loops trade reliability for autonomy; cost and latency are model-determined, so cap the steps.
- Everything ran **fully offline** and reproducibly: the lexical retriever and rule controller are the default, with the real dense-retriever and LLM-controller paths shown as the headline behind a flag.

### The Applied Track opens here

The Frontier Track closed at Part 15 with the conductor. The **Applied Track** picks up the tools and runs them: this part built the agent by hand. The best next step is the obvious one — point the loop at your own corpus and tools, watch where it loops or misroutes, and tune the controller (or hand it to a real LLM) from there.